# 🧠 AI Tutor — Self-Contained DQN Training (Colab, GPU T4)

**No git needed.** All the training code is inlined below (extracted + verified from the project). It matches the production DQN architecture (state_dim=16, 4×3×3 = **36 actions**) so the checkpoint drops straight into the app, and the eval writes the exact `eval_results.json` the app's gate reads.

**Runtime → Change runtime type → T4 GPU**, then run cells top-to-bottom.

Flow: GPU check → inline code (5 setup cells) → mount Drive → **train (time-boxed <4h, resumable)** → reward curve → eval vs baselines → verdict → download `dqn_model.pt` + `eval_results.json`.

## 0) GPU check  (set Runtime → T4 GPU)

In [ ]:
import torch
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — set Runtime->T4 GPU!")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1) Utilities — state vector, replay buffer, metrics

In [ ]:
# ---- 16-dim state vector (verbatim from utils/state_vector.py) ----
def _clamp01(x): return max(0.0, min(1.0, float(x)))
_LV_MAX, _STREAK_MAX, _CONV_TURNS_MAX, _MODE_MAX = 0.05, 10.0, 20, 3.0

def build_state_vector(knowledge, learning_velocity, confidence, concept_mastery,
                       engagement, speed, hint_dependency, streak,
                       fatigue, frustration, curiosity, focus,
                       retention, cognitive_load, conversation_turns=0, last_mode=0):
    return [
        _clamp01(knowledge), _clamp01(learning_velocity/_LV_MAX), _clamp01(confidence), _clamp01(concept_mastery),
        _clamp01(engagement), _clamp01(speed), _clamp01(hint_dependency), _clamp01(streak/_STREAK_MAX),
        _clamp01(fatigue), _clamp01(frustration), _clamp01(curiosity), _clamp01(focus),
        _clamp01(retention), _clamp01(cognitive_load),
        _clamp01(conversation_turns/_CONV_TURNS_MAX), _clamp01(last_mode/_MODE_MAX),
    ]

# ---- Replay buffer (no-DB version of models/replay_buffer.py) ----
from collections import deque
import random as _random

class ReplayBuffer:
    def __init__(self, capacity=20000, db_collection=None):
        self.buffer = deque(maxlen=capacity)
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    def sample(self, batch_size):
        batch = _random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (torch.stack(states), torch.tensor(actions),
                torch.tensor(rewards, dtype=torch.float32),
                torch.stack(next_states), torch.tensor(dones, dtype=torch.float32))
    def __len__(self): return len(self.buffer)

# ---- RL metrics (verbatim from utils/rl_metrics.py, trimmed) ----
import time as _time
class RLMetrics:
    def __init__(self, reward_window=500, loss_window=500):
        self.rewards=deque(maxlen=reward_window); self.losses=deque(maxlen=loss_window)
        self.mode_counts={0:0,1:0,2:0,3:0}; self.hint_counts={0:0,1:0,2:0}; self.difficulty_counts={}
        self.total_decisions=0; self.total_learns=0; self.total_train_steps=0
    def record_decision(self, mode, hint, difficulty, epsilon):
        self.total_decisions+=1; self.mode_counts[mode]=self.mode_counts.get(mode,0)+1
        self.hint_counts[hint]=self.hint_counts.get(hint,0)+1
        self.difficulty_counts[round(difficulty,2)]=self.difficulty_counts.get(round(difficulty,2),0)+1
    def record_loss(self, v): self.losses.append(v); self.total_train_steps+=1
    def mean_loss(self): return sum(self.losses)/len(self.losses) if self.losses else 0.0
print("utilities ready")

## 2) DQN model + agent  (verbatim from models/dqn.py — same architecture the app loads)

In [ ]:
import torch.nn as nn
import torch.optim as optim
import os

class DQN(nn.Module):
    def __init__(self, state_dim=16, action_dim=36):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim,256), nn.ReLU(),
            nn.Linear(256,256), nn.ReLU(),
            nn.Linear(256,128), nn.ReLU(),
            nn.Linear(128,action_dim))
    def forward(self, x): return self.net(x)

class DQNAgent:
    def __init__(self, action_space, db_collection=None, min_buffer=500):
        self.action_space=action_space
        self.gamma=0.95; self.epsilon=1.0; self.epsilon_decay=0.999; self.epsilon_min=0.05
        self.train_freq=4; self.min_buffer=min_buffer
        self.model=DQN(16,len(action_space)).to(device)
        self.target_model=DQN(16,len(action_space)).to(device)
        self.target_model.load_state_dict(self.model.state_dict()); self.target_model.eval()
        self.optimizer=optim.Adam(self.model.parameters(), lr=0.0003)
        self.loss_fn=nn.MSELoss()
        self.replay_buffer=ReplayBuffer(20000, db_collection=None)
        self.batch_size=64; self.target_update_freq=500; self.step_counter=0
        self.metrics=RLMetrics()

    def get_state(self, student, kt_mastery=None):
        concept=student.get_current_concept()
        knowledge=kt_mastery if kt_mastery is not None else concept.knowledge
        vec=build_state_vector(knowledge=knowledge, learning_velocity=student.learning_velocity,
            confidence=student.confidence, concept_mastery=concept.concept_mastery,
            engagement=student.engagement, speed=student.speed, hint_dependency=student.hint_dependency,
            streak=student.streak, fatigue=student.fatigue, frustration=student.frustration,
            curiosity=student.curiosity, focus=student.focus, retention=student.retention,
            cognitive_load=student.cognitive_load,
            conversation_turns=getattr(student,"conversation_turns",0), last_mode=getattr(student,"last_mode",0))
        return torch.tensor(vec, dtype=torch.float32, device=device)

    def get_action(self, student, explore=True, serve_epsilon=0.0, kt_mastery=None):
        state=self.get_state(student, kt_mastery=kt_mastery)
        eps=self.epsilon if explore else serve_epsilon
        if _random.random()<eps:
            idx=_random.randrange(len(self.action_space))
        else:
            with torch.no_grad(): idx=torch.argmax(self.model(state)).item()
        return idx, self.action_space[idx]

    def store_transition(self, state, action, reward, next_state, done=False):
        self.replay_buffer.push(torch.tensor(state,dtype=torch.float32), action, reward,
                                torch.tensor(next_state,dtype=torch.float32), done)

    def train_step(self):
        if len(self.replay_buffer)<self.min_buffer: return
        states,actions,rewards,next_states,dones=self.replay_buffer.sample(self.batch_size)
        states=states.to(device); next_states=next_states.to(device)
        actions=torch.tensor(actions,device=device); rewards=torch.tensor(rewards,dtype=torch.float32,device=device)
        dones=torch.tensor(dones,dtype=torch.float32,device=device)
        q=self.model(states).gather(1,actions.unsqueeze(1)).squeeze()
        with torch.no_grad():
            na=torch.argmax(self.model(next_states),dim=1)
            maxq=self.target_model(next_states).gather(1,na.unsqueeze(1)).squeeze()
            target=rewards+self.gamma*maxq*(1-dones)
        loss=self.loss_fn(q,target)
        self.optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(),1.0); self.optimizer.step()
        self.metrics.record_loss(loss.item()); self.step_counter+=1
        if self.step_counter%self.target_update_freq==0:
            self.target_model.load_state_dict(self.model.state_dict())
        self.epsilon=max(self.epsilon_min,self.epsilon*self.epsilon_decay)

    def save_checkpoint(self, path="checkpoints/dqn_model.pt"):
        os.makedirs(os.path.dirname(path) or "checkpoints", exist_ok=True)
        torch.save({"model":self.model.state_dict(),"target_model":self.target_model.state_dict(),
                    "optimizer":self.optimizer.state_dict(),"epsilon":self.epsilon,
                    "step_counter":self.step_counter}, path)

    def load_checkpoint(self, path="checkpoints/dqn_model.pt"):
        if not os.path.exists(path): print("No checkpoint — starting fresh."); return False
        saved=torch.load(path, map_location=device, weights_only=True)
        if isinstance(saved,dict) and "model" in saved:
            self.model.load_state_dict(saved["model"]); self.target_model.load_state_dict(saved["target_model"])
            self.optimizer.load_state_dict(saved["optimizer"]); self.epsilon=saved["epsilon"]; self.step_counter=saved["step_counter"]
            print("Resumed checkpoint step=%d eps=%.3f"%(self.step_counter,self.epsilon))
        else:
            self.model.load_state_dict(saved)
        return True
print("DQNAgent ready")

## 3) Reward math + simulated student  (reward.py verbatim; SimStudent **fixed** for full traits)

In [ ]:
import math
MODE_DIRECT_QUESTION, MODE_SOCRATIC_PROBE, MODE_REVEAL_STEP, MODE_CHALLENGE = 0,1,2,3
def smooth_clip(x): return max(0.0, min(1.0, 1.0/(1.0+math.exp(-4.0*(x-0.5)))))

def update_student_traits(traits, concept, correct, time_taken, hint_used, difficulty, mark_reviewed_fn=None):
    old_knowledge=concept["knowledge"]; old_engagement=traits["engagement"]
    lr=0.08+0.12*traits["focus"]
    if correct:
        concept["knowledge"]+=lr*(1.0-concept["knowledge"]); concept["concept_mastery"]+=0.06*(1.0-concept["concept_mastery"])
    else:
        concept["knowledge"]-=0.06*concept["knowledge"]; concept["concept_mastery"]-=0.05*concept["concept_mastery"]
    kg=concept["knowledge"]-old_knowledge; a=0.2
    traits["learning_velocity"]=(1-a)*traits["learning_velocity"]+a*kg/max(1,time_taken)
    traits["engagement"]+= 0.1*(1.0-traits["engagement"]) if correct else -0.12*traits["engagement"]
    traits["confidence"]+= 0.08*(1.0-traits["confidence"]) if correct else -0.1*traits["confidence"]
    traits["frustration"]+= -0.12*traits["frustration"] if correct else 0.15*(1.0-traits["frustration"])
    traits["streak"]=traits["streak"]+1 if correct else 0
    level="easy" if difficulty<0.3 else ("medium" if difficulty<0.5 else "hard")
    max_time={"easy":20,"medium":40,"hard":90}[level]; tr=min(1.0,time_taken/max_time)
    traits["fatigue"]+=0.04*difficulty+0.02*tr
    traits["cognitive_load"]+=0.1*difficulty+0.05*hint_used
    traits["hint_dependency"]+=0.08*hint_used
    traits["focus"]-=0.4*traits["fatigue"]; traits["engagement"]-=0.3*traits["frustration"]
    traits["curiosity"]+=0.25*traits["confidence"]; traits["retention"]-=0.2*traits["cognitive_load"]
    traits["speed"]=0.5*traits["confidence"]+0.5*(1.0-traits["fatigue"])
    concept["knowledge"]=smooth_clip(concept["knowledge"]); concept["concept_mastery"]=smooth_clip(concept["concept_mastery"])
    for k in ["learning_velocity","engagement","confidence","frustration","fatigue","focus",
              "curiosity","hint_dependency","retention","cognitive_load","speed"]:
        traits[k]=smooth_clip(traits[k])
    return old_knowledge, old_engagement

def compute_reward(traits, concept, correct, time_taken, hint, difficulty, mode, old_knowledge, old_engagement):
    kg=concept["knowledge"]-old_knowledge; ed=traits["engagement"]-old_engagement
    challenge_match=1.0-abs(difficulty-concept["knowledge"])
    hint_effect=(hint*0.05)*((1.0-concept["knowledge"])-0.5)
    if correct: speed_bonus=0.5 if time_taken<2 else (0.2 if time_taken<5 else 0.0)
    else:       speed_bonus=-0.1 if time_taken>8 else 0.0
    streak_bonus=min(0.5,0.05*traits["streak"]); mode_bonus=0.0
    if mode==MODE_SOCRATIC_PROBE:
        mode_bonus=(0.8 if (correct and hint==0) else (0.3 if correct else -0.1))+0.15*traits["engagement"]
    elif mode==MODE_REVEAL_STEP:
        mode_bonus=0.4 if correct else 0.1
    elif mode==MODE_CHALLENGE:
        mode_bonus=(1.0*difficulty if correct else 0.0) - (0.5 if traits["frustration"]>0.6 else 0.0)
    return 8.0*kg+2.0*ed+1.0*challenge_match+speed_bonus+hint_effect+streak_bonus+mode_bonus

# ---- SimStudent (FIXED: always full 14 traits; partial ranges only override) ----
_SIM_DEFAULTS = {"knowledge":[0.1,0.8],"concept_mastery":[0.1,0.6],"learning_velocity":[0.3,0.7],
    "confidence":[0.2,0.8],"engagement":[0.3,0.9],"frustration":[0.0,0.4],"streak":[0,3],
    "fatigue":[0.0,0.3],"cognitive_load":[0.1,0.4],"hint_dependency":[0.0,0.3],"focus":[0.4,0.8],
    "curiosity":[0.3,0.7],"retention":[0.4,0.8],"speed":[0.3,0.7]}

class SimStudent:
    def __init__(self, trait_ranges=None):
        ranges={**_SIM_DEFAULTS, **(trait_ranges or {})}   # merge -> never drop a trait
        self.traits={}
        for k,(lo,hi) in ranges.items():
            self.traits[k]=_random.randint(int(lo),int(hi)) if k=="streak" else _random.uniform(lo,hi)
        self.concept={"knowledge":self.traits.pop("knowledge"),"concept_mastery":self.traits.pop("concept_mastery")}
        self.conversation_turns=0; self.last_mode=0
    def respond(self, mode, hint, difficulty):
        k=self.concept["knowledge"]; t=self.traits
        p=k-difficulty+0.5 + hint*0.12 + 0.1*(t["focus"]-0.5) + 0.05*(t["confidence"]-0.5) \
          - 0.15*t["fatigue"] - 0.1*t["frustration"]
        if mode==1: p+= -0.08 if hint==0 else 0.0
        elif mode==2: p+=0.06
        elif mode==3: p-=0.12
        p=max(0.05,min(0.95,p)); correct=_random.random()<p
        bt=3.0+8.0*difficulty+5.0*t["fatigue"]-hint*1.5-3.0*k
        rt=max(1.0, bt+_random.gauss(0,1.5))
        self.conversation_turns+=1; self.last_mode=mode
        return correct, rt

# ---- action space + adapter (from train_offline.py) ----
def build_action_space():
    return [(m,h,d) for m in range(4) for h in range(3) for d in (0.2,0.4,0.6)]

class _ConceptProxy:
    def __init__(self, c): self.knowledge=c["knowledge"]; self.concept_mastery=c["concept_mastery"]

class SimStudentAdapter:
    def __init__(self, sim): self._sim=sim; self._sync_from_sim()
    def _sync_from_sim(self):
        for k,v in self._sim.traits.items(): setattr(self,k,v)
        self.conversation_turns=self._sim.conversation_turns; self.last_mode=self._sim.last_mode
    def get_current_concept(self): return _ConceptProxy(self._sim.concept)
print("reward + SimStudent ready (36 actions:", len(build_action_space()), ")")

## 4) Baseline policies + eval harness  (verbatim from baselines.py / eval_policies.py)

In [ ]:
import copy, json
MODES=[0,1,2,3]; HINTS=[0,1,2]; DIFFS=[0.2,0.4,0.6]

class RandomPolicy:
    name="Random"
    def select_action(self, traits, concept): return (_random.choice(MODES),_random.choice(HINTS),_random.choice(DIFFS))

class FixedLadderPolicy:
    name="FixedLadder"
    def select_action(self, traits, concept):
        s=traits.get("streak",0)
        return (0,2,0.2) if s<2 else ((0,1,0.4) if s<5 else (3,0,0.6))

class MasteryThresholdPolicy:
    name="MasteryThreshold"
    def select_action(self, traits, concept):
        m=concept.get("knowledge",0.5); fr=traits.get("frustration",0.0); fa=traits.get("fatigue",0.0); c=traits.get("confidence",0.5)
        if fr>0.6 or fa>0.7: return (2,2,0.2)
        if m>0.7 and c>0.5:  return (3,0,0.6)
        if m>0.4:            return (1, 1 if c<0.4 else 0, 0.4)
        return (0, 2 if c<0.3 else 1, 0.2)

class DQNPolicy:
    def __init__(self, agent, name="DQN"): self.name=name; self.agent=agent
    def select_action(self, traits, concept):
        sim=SimStudent.__new__(SimStudent)
        sim.traits=dict(traits); sim.concept=dict(concept); sim.conversation_turns=0; sim.last_mode=0
        _,action=self.agent.get_action(SimStudentAdapter(sim), explore=False, serve_epsilon=0.0)
        return action

def create_frozen_learners(n, seed=None):
    if seed is not None: _random.seed(seed)
    out=[]
    for _ in range(n):
        s=SimStudent(); out.append({"traits":copy.deepcopy(s.traits),"concept":copy.deepcopy(s.concept)})
    return out

def run_policy(policy, snaps, max_steps=50, seed=None):
    res=[]
    for i,snap in enumerate(snaps):
        if seed is not None: _random.seed(seed+i*7919)
        sim=SimStudent.__new__(SimStudent)
        sim.traits=copy.deepcopy(snap["traits"]); sim.concept=copy.deepcopy(snap["concept"])
        sim.conversation_turns=0; sim.last_mode=0
        ik=sim.concept["knowledge"]; cr=0.0; ms=None; step=0
        for step in range(max_steps):
            mode,hint,diff=policy.select_action(sim.traits, sim.concept)
            correct,rt=sim.respond(mode,hint,diff)
            ok_,oe_=update_student_traits(sim.traits,sim.concept,correct,rt,hint,diff)
            cr+=compute_reward(sim.traits,sim.concept,correct,rt,hint,diff,mode,ok_,oe_)
            if ms is None and sim.concept["knowledge"]>0.8: ms=step+1
            if sim.traits["fatigue"]>0.9 or sim.traits["frustration"]>0.9: break
        res.append({"knowledge_gain":sim.concept["knowledge"]-ik,"cumulative_reward":cr,
            "final_frustration":sim.traits["frustration"],"final_engagement":sim.traits["engagement"],
            "final_knowledge":sim.concept["knowledge"],"questions_to_mastery":ms})
    return res

def aggregate_metrics(res):
    n=len(res)
    ms=[r["questions_to_mastery"] for r in res if r["questions_to_mastery"] is not None]
    return {"n_learners":n,
        "mean_knowledge_gain":sum(r["knowledge_gain"] for r in res)/n,
        "mean_reward":sum(r["cumulative_reward"] for r in res)/n,
        "mean_frustration":sum(r["final_frustration"] for r in res)/n,
        "mean_engagement":sum(r["final_engagement"] for r in res)/n,
        "mastery_rate":len(ms)/n,
        "mean_questions_to_mastery":sum(ms)/max(len(ms),1),
        "mean_final_knowledge":sum(r["final_knowledge"] for r in res)/n}

def check_dqn_beats_baselines(results):
    dqn=[n for n in results if "DQN" in n.upper()]
    if not dqn: return False
    base=[n for n in results if n not in dqn and n.lower()!="random"]
    d=results[dqn[0]]
    return all(d["mean_reward"]>results[b]["mean_reward"] and d["mean_knowledge_gain"]>results[b]["mean_knowledge_gain"] for b in base)
print("baselines + eval harness ready")

## 5) Mount Google Drive  (checkpoints persist across disconnects / 4h limit)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR='/content/drive/MyDrive/ai_tutor_dqn'
os.makedirs(SAVE_DIR, exist_ok=True)
CKPT=os.path.join(SAVE_DIR,'dqn_model.pt')
print("Saving to:", SAVE_DIR, "| existing:", os.listdir(SAVE_DIR))

## 6) TRAIN  (GPU, time-boxed < 4h, resumable)

Re-run this cell after a disconnect — it loads the Drive checkpoint and continues. Reward should trend up.

In [ ]:
import time
from collections import deque

TIME_BUDGET_HRS = 3.5      # stop before Colab's ~4h cutoff
MAX_EPISODES    = 300000   # upper cap; time budget usually stops first
MAX_STEPS, LOG_EVERY, CKPT_EVERY, MIN_BUFFER = 50, 100, 500, 500

agent=DQNAgent(build_action_space(), db_collection=None, min_buffer=MIN_BUFFER)
agent.load_checkpoint(CKPT)
if agent.step_counter==0: agent.epsilon=1.0

win=deque(maxlen=100); hist=[]; best=float('-inf'); t0=time.time(); ep=0
print("Training on:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
while ep<MAX_EPISODES:
    ep+=1
    sim=SimStudent(); ad=SimStudentAdapter(sim); er=0.0
    for _ in range(MAX_STEPS):
        st=agent.get_state(ad).tolist()
        ai,(mode,hint,diff)=agent.get_action(ad, explore=True)
        correct,rt=sim.respond(mode,hint,diff)
        ok_,oe_=update_student_traits(sim.traits,sim.concept,correct,rt,hint,diff)
        r=compute_reward(sim.traits,sim.concept,correct,rt,hint,diff,mode,ok_,oe_)
        ad._sync_from_sim()
        ns=agent.get_state(ad).tolist()
        done=(sim.traits["fatigue"]>0.9 or sim.traits["engagement"]<0.1 or sim.traits["frustration"]>0.9)
        agent.store_transition(st,ai,r,ns,done); agent.train_step(); er+=r
        if done: break
    win.append(er); avg=sum(win)/len(win)
    if ep%LOG_EVERY==0:
        hist.append((ep,avg))
        print("Ep %6d | avg100 %8.2f | eps %.4f | buf %5d | %5.1f min"%(ep,avg,agent.epsilon,len(agent.replay_buffer),(time.time()-t0)/60))
    if ep%CKPT_EVERY==0:
        agent.save_checkpoint(CKPT)
        if avg>best: best=avg; agent.save_checkpoint(CKPT.replace('.pt','_best.pt'))
    if time.time()-t0>TIME_BUDGET_HRS*3600:
        print("Time budget reached, stopping at ep",ep); break
agent.save_checkpoint(CKPT)
print("\nDONE | episodes=%d steps=%d best_avg=%.2f eps=%.4f"%(ep,agent.step_counter,best,agent.epsilon))

## 7) Reward curve

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9,4)); plt.plot([h[0] for h in hist],[h[1] for h in hist])
plt.xlabel('episode'); plt.ylabel('avg reward (last 100)'); plt.title('DQN training reward curve'); plt.grid(True,alpha=.3); plt.show()

## 8) Evaluate vs baselines → `eval_results.json`  (the production gate file)

In [ ]:
N_LEARNERS, STEPS, SEED = 300, 50, 42
learners=create_frozen_learners(N_LEARNERS, seed=SEED)
policies=[RandomPolicy(), FixedLadderPolicy(), MasteryThresholdPolicy(), DQNPolicy(agent, name="DQN")]
results={}
for p in policies:
    print("Evaluating", p.name, "...", end=" ")
    results[p.name]=aggregate_metrics(run_policy(p, learners, max_steps=STEPS, seed=SEED))
    print("reward=%.2f"%results[p.name]["mean_reward"])

with open('eval_results.json','w') as f: json.dump(results,f,indent=2)
import shutil; shutil.copy('eval_results.json', os.path.join(SAVE_DIR,'eval_results.json'))
print("saved eval_results.json (+ Drive)")

## 9) Analysis + SHIP verdict

In [ ]:
try:
    import pandas as pd
    df=pd.DataFrame(results).T[['mean_reward','mean_knowledge_gain','mean_frustration','mean_engagement','mastery_rate','mean_questions_to_mastery']]
    display(df.round(4))
except Exception: print(json.dumps(results,indent=2))

ok=check_dqn_beats_baselines(results)
print("\n"+("✅ SHIP-READY — DQN beats FixedLadder & MasteryThreshold on reward AND knowledge gain. dqn_validated=True in prod."
             if ok else "❌ NOT beating baselines yet — raise MAX_EPISODES and re-run Cell 6, then re-eval."))
names=list(results); rew=[results[n]['mean_reward'] for n in names]
plt.figure(figsize=(8,4)); plt.bar(names,rew); plt.ylabel('mean reward'); plt.title('Policy comparison (higher=better)')
plt.xticks(rotation=20); plt.grid(True,axis='y',alpha=.3); plt.show()

## 10) Download outputs

In [ ]:
from google.colab import files
print("On Drive:", os.listdir(SAVE_DIR))
files.download(CKPT)                 # dqn_model.pt
files.download('eval_results.json')

## ✅ Next — turn the DQN on in production  *(only if SHIP-READY)*

Put the two files in your repo, then tell me — I'll wire the Dockerfile to bake them + set the flag:
```bash
cp dqn_model.pt      checkpoints/dqn_model.pt
cp eval_results.json eval_results.json
git add -f checkpoints/dqn_model.pt eval_results.json
git commit -m "rl: trained DQN checkpoint + eval (ship-ready)"
git push
```
Then Render → Environment → `RL_ENABLED=true`. `policy_tag` flips to **`dqn`** → all 36 actions go live.